In [ ]:
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as np
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import SparsePauliOp, Statevector, Operator
from qiskit.primitives import StatevectorEstimator as Estimator
from scipy.optimize import minimize
from qiskit_algorithms import NumPyMinimumEigensolver, VQE, VQD
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_algorithms.optimizers import COBYLA

# Support Vector Machines

Support Vector Machines (SVMs) are machine learning models commonly used for binary classification. Let us first assume that the dataset is linearly separable. Given a dataset with samples $x_j \in \mathbb{R}^n$ and labels $y_j \in {-1,1}$, the objective of an SVM is to find a separating hyperplane

$$
H = \{x \in \mathbb{R}^n : \vec{w}\cdot \vec{x}+b=0\}
$$

where $\vec{w} \in \mathbb{R}^n$ is the normal vector to the hyperplane and $b \in \mathbb{R}$ is the bias term.

The decision rule is based on the sign of

$$
f(x)=\vec{w}\cdot \vec{x}+b
$$

If $f(x)>0$, the sample is assigned to class $+1$, whereas if $f(x)<0$, it is assigned to class $-1$. Points satisfying $f(x)=0$ lie exactly on the decision boundary, so their classification is ambiguous unless a specific convention is chosen.

First, let's consider the hard-margin SVM, where no elements of the dataset are allowed to be inside the margin and therefore, none are misclassified. Consequently, training samples are constrained to satisfy:

$$
y_j(\vec{w}\cdot \vec{x}_j+b) \geq 1
$$

Therefore, the closest training samples do not lie on the separating hyperplane $f(x)=0$, but on the margin hyperplanes

$$
\vec{w}\cdot \vec{x}+b = 1
$$

and

$$
\vec{w}\cdot \vec{x}+b = -1
$$

These closest samples are called support vectors. In this particular case, the training of the SVM can be written as the following optimization problem:

$$
\begin{aligned}
\text{Minimize} \quad & \frac{1}{2}||w||^2 
\end{aligned}
$$
$$
\begin{aligned}
\text{subject to} \quad & y_j(\vec{w}\cdot \vec{x_j} + b) \geq 1
\end{aligned}
$$

If constrast, soft-margin SVM admit some tolerance introduced by some parameters $\xi_j \geq0$. The optimization problem for the training of soft-margin SVM can be expressed as:

$$
\begin{aligned}
\text{Minimize} \quad & \frac{1}{2}||w||^2 + C\sum_j \xi_j 
\end{aligned}
$$
$$
\begin{aligned}
\text{subject to} \quad & y_j(\vec{w}\cdot \vec{x_j} + b) \geq 1 - \xi_j, \quad & \xi_j \geq 0 
\end{aligned}
$$

Here, $C>0$ is a hyperparameter that quantifies how tolerant we are to training examples falling inside the margin or even to be missclassified. The bigger the value of $C$, the closest the model will be to the hard-margin case. The soft margin case can be equivalently written in terms of some optimizable parameters $\alpha_j$ as:

$$
\begin{aligned}
\text{Minimize} \quad & \sum_j \alpha_j - \frac{1}{2}\sum_{j,k}y_j y_k \alpha_j \alpha_k (\vec{x_j}\cdot \vec{x_k})
\end{aligned}
$$
$$
\begin{aligned}
\text{subject to} \quad & \sum_j \alpha_j y_j , \quad & 0 \leq \alpha_j \leq C
\end{aligned}
$$

The relation between these to formulations can be shown to be:

$$
\vec{w} = \sum_j \alpha_j y_j \vec{x_j}
$$

And $b$ can be recovered by finding some $\vec{x_j}$ that lies at the margin and solving a simple ecuation. Given that, we can classify points by computing:

$$
f(x)=\vec{w}\cdot \vec{x}+b = \sum_j \alpha_j y_j (\vec{x}\cdot \vec{x})+b
$$

and checking the sign of the result values, exactly as before. Notice that, in this formulation, $\vec{w}$ only depends on the points $\vec{x_j}$ for which $\alpha_j \neq 0$. Those vectors are called the support vectors.

# The kernel trick



Considering only linearly separable data results in a very limited model that is not of much practical interest. To overcome this limitation, we will use the kernel trick, which is a technique that maps the original data from its original space $\mathbb{R}^n$ to a higher dimensional space $\mathbb{R}^N$ called feature space. The reason for this mapping is that we hope that the data will be separable by a hyperplane in the feature space. In order to perform the mapping, we will use the function:

$$
\phi: \mathbb{R}^n \rightarrow \mathbb{R}^N 
$$

known as feature map.

In order to train the SVM and to classify data, we don't need to know the exact form of the feature map, we only need to be able to compute scalar products of elemets returned by the feature map. The erasn for this, is that in both training and classification processes, the only operation that depends on the $\vec{x_j}$ points is the inner product $\vec{x_j}\cdot\vec{x_k}$ when training, or $\vec{x_j}\cdot\vec{x}$ when classifying $\vec{x}$. That way, we just need to be able to compute functions of the form:

$$
k(x,y) = \phi (\vec{x})\cdot\phi (\vec{y})
$$

known as kenel functions.

# Quantum Support Vector Machines

Quantum Support Vector Machines (QSVM) are a particular case of SVM where the feature space is a specific space of quantum states, the Hilbert space.